In [4]:
from pathlib import Path
from shutil import which

# 作業ディレクトリ（main.jl がある場所）
WORKDIR = Path("/home/mori-lab/shimizu/ACO-on-Network/src/test").resolve()
MAIN_FILE = WORKDIR / "main.jl"

# Julia 実行パスを解決
JULIA_CMD = which("julia")
if JULIA_CMD is None:
    # よくある場所を当たる（必要なら手動で指定してください）
    candidates = [
        "/usr/bin/julia",
        "/usr/local/bin/julia",
        str(Path.home()/".juliaup/bin/julia"),
    ]
    for c in candidates:
        p = Path(c)
        if p.exists():
            JULIA_CMD = str(p)
            break

print("WORKDIR:", WORKDIR)
print("MAIN_FILE exists:", MAIN_FILE.exists())
print("JULIA_CMD:", JULIA_CMD)


WORKDIR: /home/mori-lab/shimizu/ACO-on-Network/src/test
MAIN_FILE exists: True
JULIA_CMD: /home/mori-lab/.juliaup/bin/julia


In [ ]:
import subprocess, re

def run_julia_main(*, N=100, T=200_000, r=100, omega=-0.9999, alpha=0.8, h=0.001, J=0.1, samples=100, parallel=True):
    if JULIA_CMD is None:
        raise RuntimeError("Julia 実行ファイルが見つかりません（セル1で JULIA_CMD を設定してください）")
    if not MAIN_FILE.exists():
        raise FileNotFoundError(f"main.jl が見つかりません: {MAIN_FILE}")

    cmd = [JULIA_CMD]
    if parallel:
        cmd += ["-p", "auto"]
    cmd += [
        str(MAIN_FILE.name),  # cwd を WORKDIR にしているのでファイル名のみでOK
        "--N", str(N),
        "--T", str(T),
        "--r", str(r),
        "--omega", str(omega),
        "--alpha", str(alpha),
        "--h", str(h),
        "--J", str(J),
        "--sample", str(samples),
    ]
    print("Running:", " ".join(cmd))
    res = subprocess.run(cmd, cwd=str(WORKDIR), capture_output=True, text=True)
    if res.returncode != 0:
        print("STDOUT:\n", res.stdout)
        print("STDERR:\n", res.stderr)
        raise RuntimeError(f"main.jl failed (alpha={alpha}, omega={omega})")
    return True


In [ ]:
from itertools import product

# 既定（main.jlのデフォルトと同じにしてあります。必要に応じて変更）
N = 100
T = 200_000
r = 100
h = 0.001
J = 0.1
samples = 100

# ここでパラメータを振る
alphas = [0.5, 0.7, 0.8, 0.85, 0.9, 0.99]
omegas = [-1.0, -0.9999, -0.999, -0.99, -0.9, 0.0, 1.0]

# 一括実行
for alpha, omega in product(alphas, omegas):
    run_julia_main(N=N, T=T, r=r, omega=omega, alpha=alpha, h=h, J=J, samples=samples)

print("All jobs finished.")


Running: julia -p auto main.jl --N 100 --T 200000 --r 100 --omega -1.0 --alpha 0.5 --h 0.001 --J 0.1 --sample 100


FileNotFoundError: [Errno 2] No such file or directory: 'julia'

In [3]:
pwd


'/home/mori-lab/shimizu/ACO-on-Network/src/test'